# Trabalho 2 — 03 · RAG Chatbot fundamentado
### Pokédex Inteligente · 151 Pokémon de Kanto / FireRed

**Objetivo:** usar o índice TF-IDF do Notebook 02 para recuperar documentos e, **somente depois**, utilizar um LLM para formular respostas em português com evidências auditáveis.

**Pipeline acadêmico:** Trabalho 1 (corpus + PLN tradicional) → Notebook 01 (documentos temáticos) → Notebook 02 (TF-IDF + avaliação) → **Notebook 03 (RAG + avaliação da geração)**.

**Restrições:** sem novo treinamento, sem nova coleta, sem indexação e sem busca web. Modelo generativo **não substitui** o corpus. Uma resposta textual do LLM não é automaticamente considerada factual: o sistema exige citações a documentos efetivamente recuperados e trechos literalmente presentes neles. Mesmo assim, validação de IDs e citações **não prova** que a afirmação seja logicamente sustentada; esse aspecto será objeto de revisão humana.

> **Execução:** o FLAN-T5 roda localmente e lê somente um cache local previamente provisionado. O provisionamento é uma etapa explícita de configuração, separada da resposta do chatbot. Os experimentos sem corpus são **exclusivos de avaliação** e nunca alimentam o chatbot de produção.
**Atualização:** memória de múltiplos Pokémon, pedidos de esclarecimento e respostas contextuais a limites.


## 0 · Configuração e diretórios
Usamos os mesmos diretórios do Notebook 02. No Colab, as dependências ausentes e o modelo `google/flan-t5-small` são provisionados automaticamente nesta etapa de configuração; a geração continua lendo somente o cache local. Fora do Colab, defina `PROVISION_FLAN_MODEL=1` para provisionar o modelo e use `PLN_AVALIACAO_DIR` apenas quando precisar apontar explicitamente para outra pasta.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from IPython.display import display, HTML, Markdown
import html
import hashlib
import json
import logging
import os
import re
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DRIVE_AVAILABLE = False

if IN_COLAB:
    try:
        drive.mount('/content/drive', timeout_ms=120000)
        DRIVE_AVAILABLE = True
    except Exception as drive_error:
        print(f'Google Drive indisponível; continuando com arquivos locais: {type(drive_error).__name__}')

def resolve_workspace_root():
    """Resolve a raiz do monorepo atual ou da estrutura legada no Drive."""
    configured = os.environ.get('PLN_AVALIACAO_DIR', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / 'backend').is_dir() and (candidate / 'notebooks').is_dir() and (candidate / 'data').is_dir():
            return candidate
    if IN_COLAB and DRIVE_AVAILABLE:
        return Path('/content/drive/MyDrive/IFSP-ADS/ultimo_periodo/tc_2/avaliacao')
    for candidate in candidates:
        if (candidate / 'application' / 'data' / 'rag' / 'documents').is_dir():
            return candidate
    return Path.cwd().resolve()

AVALIACAO_DIR = resolve_workspace_root()
MONOREPO_LAYOUT = (AVALIACAO_DIR / 'backend').is_dir() and (AVALIACAO_DIR / 'data').is_dir()
if MONOREPO_LAYOUT:
    APPLICATION_DIR = AVALIACAO_DIR
    DATA_DIR = AVALIACAO_DIR / 'data'
    DOC_DIR = DATA_DIR / 'processed'
    ARTIFACTS_DIR = DATA_DIR / 'artifacts' / 'retrieval'
    REPORTS_DIR = DATA_DIR / 'reports'
    LOGS_DIR = DATA_DIR / 'logs'
    EVAL_DIR = DATA_DIR / 'evaluation'
else:
    APPLICATION_DIR = AVALIACAO_DIR / 'application'
    DATA_DIR = APPLICATION_DIR / 'data'
    DOC_DIR = DATA_DIR / 'rag' / 'documents'
    ARTIFACTS_DIR = APPLICATION_DIR / 'artifacts' / 'retrieval'
    REPORTS_DIR = APPLICATION_DIR / 'reports'
    LOGS_DIR = APPLICATION_DIR / 'logs'
    EVAL_DIR = DATA_DIR / 'evaluation'
for directory in (REPORTS_DIR, LOGS_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DOCUMENTS_PATH = DOC_DIR / 'pokemon_rag_documents.jsonl'
RECORDS_PATH = DOC_DIR / 'pokemon_rag_records.jsonl'
MANIFEST_PATH = ARTIFACTS_DIR / 'retrieval_manifest.json'
RUN_AT = datetime.now(timezone.utc).isoformat()

logger = logging.getLogger('pokemon_rag_chatbot')
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.propagate = False
logger.addHandler(logging.FileHandler(LOGS_DIR / '03_rag_chatbot.log', encoding='utf-8'))
# logger.addHandler(logging.StreamHandler())
print('Documentos:', DOCUMENTS_PATH)
print('Índices do Notebook 02:', ARTIFACTS_DIR)


Documentos: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\processed\pokemon_rag_documents.jsonl
Índices do Notebook 02: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\artifacts\retrieval


## 1 · Abrir somente os artefatos persistidos no Notebook 02
Conferimos hash SHA-256 do JSONL, ordenação dos IDs e dimensões. Se os documentos mudaram, é necessário reexecutar o Notebook 02. Só carregue arquivos `.joblib` gerados pelo seu próprio projeto: deserialização Joblib de origem desconhecida é insegura.

In [2]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}. Execute os notebooks anteriores.')
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Manifesto não encontrado: {MANIFEST_PATH}. Execute a exportação do Notebook 02.'
    )
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_manifest = {'source_document_sha256', 'document_ids', 'models', 'query_normalization_rules'}
missing = required_manifest - manifest.keys()
if missing:
    raise ValueError(f'Manifesto incompleto. Campos ausentes: {sorted(missing)}')

actual_sha = hashlib.sha256(DOCUMENTS_PATH.read_bytes()).hexdigest()
if actual_sha != manifest['source_document_sha256']:
    raise ValueError(
        'O documento JSONL foi alterado desde o ajuste do TF-IDF. '
        'Reexecute o Notebook 02 com os documentos atuais antes de gerar respostas.'
    )

raw_documents = read_jsonl(DOCUMENTS_PATH)
records = read_jsonl(RECORDS_PATH)
EDITORIAL_DESCRIPTION_BY_ID = {
    6: 'Suas asas conseguem levá-lo a uma altitude de aproximadamente 1.400 metros. Ele expele fogo em temperaturas muito altas.',
    25: 'Pikachu possui pequenas bolsas elétricas nas duas bochechas. Quando se sente ameaçado, libera cargas elétricas por meio delas.',
    133: 'Eevee é um Pokémon extremamente raro que pode evoluir de várias maneiras, dependendo dos estímulos que recebe.',
    149: 'Dragonite consegue voar apesar do corpo grande e volumoso. É capaz de dar a volta ao mundo em apenas 16 horas.',
}
EDITORIAL_PATH = DATA_DIR / 'reference' / 'description_translations_reviewed_ptbr.json'
if not EDITORIAL_PATH.exists():
    EDITORIAL_PATH = AVALIACAO_DIR / 'data' / 'reference' / 'description_translations_reviewed_ptbr.json'
if EDITORIAL_PATH.exists():
    EDITORIAL_DESCRIPTION_BY_ID.update({int(key): value for key, value in json.loads(EDITORIAL_PATH.read_text(encoding='utf-8')).items()})
by_id = {d['document_id']: d for d in raw_documents}
if len(by_id) != len(raw_documents):
    raise ValueError('Existem document_id duplicados no JSONL.')

ordered_ids = manifest['document_ids']
if set(ordered_ids) != set(by_id) or len(ordered_ids) != len(raw_documents):
    raise ValueError('IDs de documentos no manifesto não correspondem ao JSONL atual.')
documents = [by_id[doc_id] for doc_id in ordered_ids]
if len({int(r['id']) for r in records}) != 151:
    raise ValueError('Esperados 151 Pokémon (IDs únicos) na Pokédex de Kanto.')

model_info = manifest['models']['tfidf']
vectorizer_path = ARTIFACTS_DIR / model_info['vectorizer']
matrix_path = ARTIFACTS_DIR / model_info['matrix']
for p in (vectorizer_path, matrix_path):
    if not p.is_file():
        raise FileNotFoundError(f'Índice ausente: {p}. Reexecute o Notebook 02.')

vectorizer = joblib.load(vectorizer_path)  # apenas artefatos gerados pelo próprio projeto
index_matrix = sparse.load_npz(matrix_path)
if index_matrix.shape != (len(documents), len(vectorizer.get_feature_names_out())):
    raise ValueError('Dimensões do índice incompatíveis com documentos/vetorizador.')

DOC_LOOKUP = {d['document_id']: d for d in documents}
POKEMON_NAMES = {str(r.get('name', r.get('api_name',''))).casefold() for r in records}
print(f'{len(records)} Pokémon | {len(documents)} documentos temáticos | TF-IDF {index_matrix.shape}')
logger.info('Artefatos carregados e verificados | documentos=%d | checksum_ok', len(documents))

151 Pokémon | 1732 documentos temáticos | TF-IDF (1732, 13842)


## 2 · Normalização idêntica ao Notebook 02
Importamos o arquivo de equivalências sem recalcular vocabulários ou reexecutar o Trabalho 1.

In [3]:
rules_path = ARTIFACTS_DIR / manifest['query_normalization_rules']
if not rules_path.is_file():
    raise FileNotFoundError(f'Regras ausentes: {rules_path}')
normalization_config = json.loads(rules_path.read_text(encoding='utf-8'))
ALIAS_RULES = [('tipo pedra', 'type rock'), ('tipo rocha', 'type rock')] + [(item['from'], item['to']) for item in normalization_config['rules']]

def normalize_plain(text):
    text = unicodedata.normalize('NFKD', str(text or '').casefold())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace('-', ' ')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', text)).strip()

def normalize_query(text):
    text = normalize_plain(text)
    for source, target in ALIAS_RULES:
        text = re.sub(r'(?<!\w)' + re.escape(source) + r'(?!\w)', target, text)
    return re.sub(r'\s+', ' ', text).strip()

assert normalize_query('Pokémon do tipo pedra').endswith('type rock')
assert normalize_query('Pedra evolutiva para Eevee').startswith('evolution stone')
assert normalize_query('Rock type Pokémon').endswith('rock type pokemon')
print('Normalização compartilhada carregada do Notebook 01/02.')


Normalização compartilhada carregada do Notebook 01/02.


## 3 · Recuperador TF-IDF existente
Usa **o índice TF-IDF do Notebook 02** e similaridade de cosseno. Nenhum `fit()` ou nova coleta é executado aqui.

In [4]:
MAX_INPUT_TOKENS = 384
MAX_DOC_CHARS = 3500

def prompt_text(question, context):
    return ("Responda apenas com base no contexto. Se faltar informação, responda Não sei. "
            "Responda no idioma da pergunta, sem inventar fatos.\n"
            "Responda em uma única frase curta. Para um único fato, mencione o "
            "Pokémon, o atributo e o valor; não liste outros atributos.\n"
            f"Contexto:\n{context}\n\nPergunta: {question}\nResposta:")

def token_count(tokenizer, text):
    encoded = tokenizer(text, add_special_tokens=True, truncation=False)
    ids = encoded.input_ids if hasattr(encoded, 'input_ids') else encoded['input_ids']
    return len(ids[0] if ids and isinstance(ids[0], (list, tuple)) else ids)

def split_passages(content):
    passages, transition = [], []
    def flush_transition():
        if transition:
            passages.append('\n'.join(transition))
            transition.clear()
    for line in str(content or '').splitlines():
        line = line.strip()
        if not line:
            continue
        if re.match(r'^-\s*[a-z0-9-]+\s*->\s*[a-z0-9-]+', line, re.I):
            flush_transition()
            transition.append(line)
            continue
        if transition and re.match(r'^(?:conditions?:|not specifically verified)', line, re.I):
            transition.append(line)
            continue
        flush_transition()
        passages.extend(part.strip() for part in re.split(r'(?<=[.!?])\s+', line) if part.strip())
    flush_transition()
    return passages

def compact_evolution_passage(passage):
    match = re.search(
        r'-\s*([a-z0-9-]+)\s*->\s*([a-z0-9-]+)\.\s*(?:Conditions:\s*\((.*?)\))?',
        str(passage), re.I | re.S)
    if not match:
        return passage
    origin, target, conditions = match.groups()
    conditions = conditions or ''
    item = re.search(r'\bitem:\s*([^;.)]+)', conditions, re.I)
    level = re.search(r'\bminimum level:\s*(\d+)', conditions, re.I)
    trigger = re.search(r'\btrigger:\s*([^;.)]+)', conditions, re.I)
    relevant = []
    if item:
        relevant.append(f'item: {item.group(1).strip()}')
    if level:
        relevant.append(f'minimum level: {level.group(1)}')
    if trigger and not relevant:
        relevant.append(f'trigger: {trigger.group(1).strip()}')
    suffix = f" Conditions: ({'; '.join(relevant)})." if relevant else ''
    return f'- {origin} -> {target}.{suffix}'

def balance_rows_by_pokemon(rows):
    buckets, order = {}, []
    for row in rows:
        pokemon_id = int(row['pokemon_id'])
        if pokemon_id not in buckets:
            buckets[pokemon_id] = []
            order.append(pokemon_id)
        buckets[pokemon_id].append(row)
    balanced, offset = [], 0
    while any(offset < len(buckets[pokemon_id]) for pokemon_id in order):
        for pokemon_id in order:
            if offset < len(buckets[pokemon_id]):
                balanced.append(buckets[pokemon_id][offset])
        offset += 1
    return balanced

def rank_passages(question, content):
    normalized_question = normalize_plain(question)
    terms = {token for token in normalized_question.split() if len(token) > 2}
    passages = split_passages(content)
    requested_stat = requested_stat_name(question) if 'requested_stat_name' in globals() else None
    if requested_stat:
        stat_passages = [
            passage for passage in passages
            if re.match(rf'^-\s*{re.escape(requested_stat)}\s*:', passage, re.I)
        ]
        if stat_passages:
            return stat_passages
    evolution_query = bool(re.search(r'\b(?:evolu\w*|evolv\w*)\b', normalized_question))
    if evolution_query:
        passages = [compact_evolution_passage(passage) for passage in passages]
    def relevance(item):
        index, passage = item
        normalized = normalize_plain(passage)
        overlap = len(terms & set(normalized.split()))
        evolution = 20 if evolution_query and '->' in passage else 0
        conditions = 5 if evolution_query and re.search(r'\b(?:minimum level|item|trigger)\b', normalized) else 0
        return (-(overlap * 10 + evolution + conditions), index)
    return [passage for _, passage in sorted(enumerate(passages), key=relevance)]

def build_context(rows, question=None, tokenizer=None, max_chars=13500, max_doc_chars=MAX_DOC_CHARS):
    if question is not None and tokenizer is not None:
        ordered = balance_rows_by_pokemon(rows)
        entries = []

        def render(candidate_entries):
            return '\n\n'.join(
                f"FONTE [{entry['row']['document_id']}] | Pokémon: {entry['row']['pokemon_name']} | "
                f"seção: {entry['row']['section']}\n" + '\n'.join(entry['passages'])
                for entry in candidate_entries if entry['passages'])

        def add_passage(row, passage):
            entry = next((item for item in entries if item['row']['document_id'] == row['document_id']), None)
            created = entry is None
            if created:
                entry = {'row': row, 'passages': []}
                entries.append(entry)
            entry['passages'].append(passage)
            if token_count(tokenizer, prompt_text(question, render(entries))) > MAX_INPUT_TOKENS:
                entry['passages'].pop()
                if created:
                    entries.remove(entry)
                return False
            return True

        first_by_pokemon, seen = [], set()
        for row in ordered:
            pokemon_id = int(row['pokemon_id'])
            if pokemon_id not in seen:
                seen.add(pokemon_id)
                first_by_pokemon.append(row)
        for row in first_by_pokemon:
            for passage in rank_passages(question, row['content']):
                if add_passage(row, passage):
                    break
        for row in ordered:
            existing = next((item for item in entries if item['row']['document_id'] == row['document_id']), None)
            selected = set(existing['passages']) if existing else set()
            for passage in rank_passages(question, row['content']):
                if passage not in selected and add_passage(row, passage):
                    selected.add(passage)
        context = render(entries)
        used = [{**entry['row'], 'content': '\n'.join(entry['passages'])}
                for entry in entries if entry['passages']]
        return context, used

    parts, used, total = [], [], 0
    for row in rows:
        header = f"FONTE [{row['document_id']}] | Pokémon: {row['pokemon_name']} | seção: {row['section']}\n"
        budget = min(max_doc_chars, max_chars - total - len(header) - 2)
        if budget < 100:
            break
        content = str(row['content'])[:budget]
        parts.append(header + content)
        used.append({**row, 'content': content})
        total += len(header) + len(content) + 2
    return '\n\n'.join(parts), used

print('Contexto FLAN-T5: orçamento de entrada=', MAX_INPUT_TOKENS, 'tokens; seleção sem truncamento silencioso.')


Contexto FLAN-T5: orçamento de entrada= 384 tokens; seleção sem truncamento silencioso.


## 4 · Política de recuperação e interpretação
O ranking é calculado pelo TF-IDF persistido no Notebook 02. Metadados dos documentos orientam a seleção da seção, uma fonte é reservada por espécie em perguntas múltiplas e regras conservadoras identificam limites e perguntas que necessitam de critério.


In [5]:
import difflib
import re

REFUSAL = (
    "Não encontrei informações suficientes no corpus "
    "de Kanto/FireRed para responder com segurança."
)

NAME_TO_ID = {}
for record in records:
    name = str(record.get("api_name") or record.get("name") or "").strip()
    if name:
        NAME_TO_ID[normalize_plain(name)] = int(record["id"])

KNOWN_NAMES = sorted(NAME_TO_ID, key=len, reverse=True)
ONE_WORD_NAMES = sorted(n for n in KNOWN_NAMES if len(n.split()) == 1 and len(n) >= 4)
ID_TO_NAME = {
    int(record['id']): str(record.get('name') or record.get('api_name') or record['id']).strip()
    for record in records
}

def pokemon_names_by_ids(pokemon_ids):
    return [ID_TO_NAME[pokemon_id] for pokemon_id in pokemon_ids if pokemon_id in ID_TO_NAME]

def detect_response_language(question):
    normalized = normalize_plain(question)
    english_markers = r'\b(?:what|which|where|when|how|who|tell|about|evolve|evolution|type|level)\b'
    portuguese_markers = r'\b(?:qual|quais|onde|quando|como|quem|fale|sobre|evolui|evolucao|tipo|nivel)\b'
    if re.search(english_markers, normalized) and not re.search(portuguese_markers, normalized):
        return 'en'
    return 'pt'

COMMON_WORDS = {
    "sobre", "como", "quem", "quais", "qual",
    "quando", "onde", "porque", "para", "mais",
    "menos", "evoluir", "evolucao", "evolucoes",
    "evolui", "pokemon", "pokemons", "pokedex",
    "fire", "red", "habilidade", "habilidades",
    "estatisticas", "ataques", "golpes",
    "tipos", "tipo", "fogo", "agua", "voador",
    "pedra", "eletrico", "nome", "explique",
    "quero", "me", "diga", "have", "tell",
    "what", "which", "where", "from", "about",
    "level", "type", "evolution"
}

def canonicalize_question(question):
    normalized_question = normalize_plain(question)
    protected_tokens = set()
    for name in KNOWN_NAMES:
        if ' ' in name and re.search(r'(?<!\w)' + re.escape(name) + r'(?!\w)', normalized_question):
            protected_tokens.update(name.split())
    tokens = normalized_question.split()
    result = []
    corrections = []
    for token in tokens:
        if token in protected_tokens or token in NAME_TO_ID or token in COMMON_WORDS or len(token) < 4:
            result.append(token)
            continue
        choices = difflib.get_close_matches(token, ONE_WORD_NAMES, n=2, cutoff=0.84)
        if choices:
            top = choices[0]
            first = difflib.SequenceMatcher(None, token, top).ratio()
            second = difflib.SequenceMatcher(None, token, choices[1]).ratio() if len(choices) > 1 else 0.0
            if first - second >= 0.06 and abs(len(token) - len(top)) <= 2:
                result.append(top)
                corrections.append({"from": token, "to": top})
                continue
        result.append(token)
    return "  ".join(result), corrections

OUT_OF_SCOPE_PATTERNS = [
    r"\b(ginasio|ginasios|gym|gyms|elite four|elite 4|detonado|walkthrough|historia principal|storyline)\b"
]

SECTION_TERMS = [
    ("type_matchups", (
        "fraqueza", "fraquezas", "fraco contra",
        "resistencia", "resistencias",
        "imunidade", "imune", "weakness",
        "weak", "resistant", "effectiveness",
        "super effective"
    )),
    ("evolution", (
        "evolucao", "evolucoes", "evolui", "evoluir", "evoluiria",
        "evolves", "evolve", "evolution", "pedra evolutiva",
        "pre evolucao", "pre-evolucao", "preevolucao", "de quem evolui",
        "forma inicial", "forma evoluida", "evolucao anterior",
        "anterior form", "pre-evolution", "previous form", "evolved from"
    )),
    ("encounters_strict", (
        "onde encontro na natureza", "encontra na natureza",
        "onde aparece selvagem", "onde aparece na natureza"
    )),
    ("encounters", (
        "onde encontro", "onde encontrar", "onde capturar", "onde aparece",
        "where to find", "where catch", "wild encounter", "localizacao selvagem",
        "localizacao", "captura", "capturar", "encontrar", "localizar"
    )),
    ("obtainment", (
        "como obter", "como conseguir", "como ganhar", "como receber", "obter", "obtain", "gift", "troca",
        "disponivel no jogo", "available in firered", "ganha", "recebe"
    )),
    ("stats", (
        "velocidade", "speed", "ataque", "attack", "defesa", "defense",
        "hp", "stat", "estatistica", "estatisticas", "atributo", "atributos",
        "atributos base", "atributos-base", "status", "mais rapido", "mais forte"
    )),
    ("abilities", (
        "habilidade", "habilidades", "ability", "abilities"
    )),
    ("moves", (
        "golpe", "golpes", "movimento", "movimentos", "move", "moves",
        "tm", "hm", "aprende", "learns"
    )),
    ("profile", (
        "tipo", "tipos", "types", "type",
        "peso", "weight", "altura", "height",
        "habitat", "descricao", "description",
        "pokedex", "quem e", "o que e",
        "me explique", "explique", "fale sobre",
        "me fale", "conte sobre", "me conte",
        "informacoes sobre", "tell me about",
        "what is", "who is", "overview"
    ))
]

def explicitly_mentioned_ids(question):
    q, _ = canonicalize_question(question)
    return list(dict.fromkeys(
        NAME_TO_ID[name]
        for name in KNOWN_NAMES
        if re.search(r"(?<!\w)" + re.escape(name) + r"(?!\w)", q)
    ))

def requested_section(question):
    q = normalize_plain(question)
    if re.search(r'\b(?:atribut\w*|atronut\w*|estatistic\w*|stat\w*)\b', q):
        return 'stats'
    if re.search(r'\b(?:evolu\w*|evolv\w*|pre[- ]?evolu\w*)\b', q):
        return 'evolution'
    if re.search(r'\b(?:onde|where)\b.*\b(?:encontr\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:como|how)\b.*\b(?:encontr\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:encontr\w*|localiz\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:obter|obten\w*|conseguir|receber|ganhar|obtain|gift|trade|get)\b', q):
        return 'obtainment'
    for section, expressions in SECTION_TERMS:
        if any(re.search(r'(?<!\w)' + re.escape(expr) + r'(?!\w)', q) for expr in expressions):
            return 'encounters' if section == 'encounters_strict' else section
    return None
def scope_gate(question):
    """Regras conservadoras anteriores ao TF-IDF para limites conhecidos do corpus."""
    q = normalize_plain(question)
    if any(re.search(pattern, q) for pattern in OUT_OF_SCOPE_PATTERNS):
        return "out_of_scope"
    # A base carrega apenas os registros de números 001 a 151.
    dex = re.search(r"\b(?:pokemon|pokedex|numero|number|n|no)\s*(?:de\s+numero\s*)?(\d{1,4})\b", q)
    if dex and int(dex.group(1)) not in range(1, 152):
        return "outside_kanto_dex"
    if re.search(r"\b(?:mais fortes?|melhores? pokemons?|strongest|best pokemons?)\b", q) and not re.search(
        r"\b(?:ataqu\w*|defes\w*|velocidade|hp|attack|defense|speed|stat|atributo)\b", q
    ):
        return "needs_comparison_criterion"
    # Reconhece pedido de localização de uma espécie nomeada mas ausente no corpus.
    if not explicitly_mentioned_ids(question):
        location = re.search(
            r"\b(?:onde (?:encontro|encontrar|capturo|capturar|acho)|"
            r"como (?:consigo|conseguir|obtenho|obter)|"
            r"where (?:can i )?(?:find|catch)|how (?:can i |to )?get)\s+"
            r"(?:(?:um|uma|o|a|an|the)\s+)?([a-z0-9-]+)\s*$", q
        )
        if location and location.group(1) not in {"pokemon", "pokemons", "monstro", "pokemonzinho"}:
            return "unknown_species"
    return None


def out_of_scope(question):
    return scope_gate(question) in {"out_of_scope", "outside_kanto_dex"}


def same_type_evidence(question, targets, top_k):
    if len(targets) != 1:
        return [], 'same_type_requires_one_reference'
    target_id = targets[0]
    profiles = [row for row in documents if row['section'] == 'profile']
    target = next((row for row in profiles if int(row['pokemon_id']) == target_id), None)
    if not target:
        return [], 'missing_target_profile'
    match = re.search(r'Types in Generation III:\s*([^.]*)\.', str(target['content']), re.IGNORECASE)
    target_types = frozenset(normalize_plain(value) for value in match.group(1).split(',')) if match else frozenset()
    if not target_types:
        return [], 'missing_target_types'
    matches = [row for row in profiles if int(row['pokemon_id']) != target_id and
               (found := re.search(r'Types in Generation III:\s*([^.]*)\.', str(row['content']), re.IGNORECASE)) and
               frozenset(normalize_plain(value) for value in found.group(1).split(',')) == target_types]
    matches.sort(key=lambda row: (int(row['pokemon_id']), row['document_id']))
    return [dict(row, rank=index + 1, score=float(row.get('score', 0.0)))
            for index, row in enumerate([target] + matches[:max(1, top_k - 1)])], 'ready'

def is_type_listing_question(question):
    normalized = normalize_plain(question)
    return bool(re.search(r'\b(?:quais|liste|listar|todos?|todas?|which|list|all)\b', normalized)
                and re.search(r'\b(?:pokemon|pokemons)\b', normalized)
                and (re.search(r'\b(?:tipo|types?)\b', normalized) or requested_type(normalized)))

def requested_type(question):
    normalized = normalize_plain(question)
    aliases = {'fogo':'fire','agua':'water','planta':'grass','eletrico':'electric','voador':'flying',
               'venenoso':'poison','terrestre':'ground','pedra':'rock','inseto':'bug','fantasma':'ghost',
               'aco':'steel','lutador':'fighting','psiquico':'psychic','gelo':'ice','dragao':'dragon',
               'sombrio':'dark','normal':'normal','fire':'fire','water':'water','grass':'grass',
               'electric':'electric','flying':'flying','poison':'poison','ground':'ground','rock':'rock',
               'bug':'bug','ghost':'ghost','steel':'steel','fighting':'fighting','psychic':'psychic',
               'ice':'ice','dragon':'dragon','dark':'dark'}
    return next((value for alias, value in aliases.items() if re.search(r'(?<!\w)' + re.escape(alias) + r'(?!\w)', normalized)), None)

def is_ranking_question(question):
    normalized = normalize_plain(question)
    return bool(re.search(r'\b(?:maior(?:es)?|menor(?:es)?|melhor(?:es)?|pior(?:es)?|ranking|rank|top)\b', normalized)
                and requested_stat_name(question))

def ranking_limit(question):
    normalized = normalize_plain(question)
    if re.search(r'\b(?:todos?|all)\b', normalized): return None
    match = re.search(r'\btop\s*(\d{1,3})\b', normalized)
    return int(match.group(1)) if match else 10

def select_type_evidence(question):
    type_name = requested_type(question)
    if not type_name:
        return [], 'type_not_recognized'
    profiles = []
    for row in documents:
        if row.get('section') != 'profile': continue
        match = re.search(r'Types in Generation III:\s*([^.]*)\.', str(row.get('content', '')), re.I)
        if match and type_name in {normalize_plain(value) for value in match.group(1).split(',')}:
            profiles.append(dict(row, score=1.0))
    profiles.sort(key=lambda row: int(row['pokemon_id']))
    return [dict(row, rank=index + 1) for index, row in enumerate(profiles)], 'ready'

MIN_EVIDENCE_SCORE = float(os.environ.get('RAG_MIN_SCORE', '0.15'))

def select_evidence(question, top_k=5):
    gate = scope_gate(question)
    if gate:
        return [], gate

    canonical, _ = canonicalize_question(question)
    targets = explicitly_mentioned_ids(canonical)
    section = requested_section(canonical)
    if targets and section == 'stats' and re.search(r'\bevolu\w*\b', normalize_plain(question)):
        related_ids = set(targets)
        for row in documents:
            if row.get('section') != 'evolution': continue
            for step in evolution_transitions([row]):
                target_names = {normalize_plain(item.get('name', '')) for item in records if int(item['id']) in targets}
                if normalize_plain(step['origin']) in target_names:
                    related = next((int(item['id']) for item in records if normalize_plain(item.get('name', '')) == normalize_plain(step['target'])), None)
                    if related: related_ids.add(related)
        targets = list(dict.fromkeys([*targets, *sorted(related_ids - set(targets))]))
    if re.search(r'\b(?:mesm[oa]s?|igual(?:es)?|same)\b.*\b(?:tipo|types?)\b', normalize_plain(question)):
        return same_type_evidence(question, targets, top_k)
    if is_type_listing_question(canonical):
        return select_type_evidence(canonical)
    if is_ranking_question(canonical):
        stat = requested_stat_name(canonical)
        candidates = []
        for row in documents:
            if row.get('section') != 'stats': continue
            match = re.search(rf'-\s*{re.escape(stat)}:\s*(\d+(?:\.\d+)?)', str(row.get('content', '')), re.I)
            if match: candidates.append((float(match.group(1)), row))
        candidates.sort(key=lambda item: (-item[0], int(item[1]['pokemon_id'])))
        return [dict(row, rank=index + 1, score=float(row.get('score', 1.0))) for index, (_, row) in enumerate(candidates)], 'ready'
    if targets and section is None:
        section = "profile"

    # TF-IDF existente continua sendo o recuperador: sem embeddings e sem novo fit.
    ranked = retrieve(canonical, top_k=len(documents))
    if not ranked:
        return [], "no_lexical_match"
    if section == "encounters_strict":
        ranked = [doc for doc in ranked if doc["section"] == "encounters"]
    elif section in ("obtainment", "encounters"):
        # Evolução não comprova onde o Pokémon é encontrado ou obtido.
        ranked = [doc for doc in ranked if doc["section"] in ("obtainment", "encounters")]
    elif section:
        ranked = [doc for doc in ranked if doc["section"] == section]

    minimum_score = MIN_EVIDENCE_SCORE
    if targets:
        ranked = [doc for doc in ranked if int(doc["pokemon_id"]) in targets]
        found_targets = {int(doc["pokemon_id"]) for doc in ranked if float(doc.get("score", 0.0)) >= minimum_score}
        # Complementa consultas comparativas com TF-IDF por alvo, sem novo fit.
        for target_id in targets:
            if target_id in found_targets and section not in {'obtainment', 'encounters'}:
                continue
            record = next((item for item in records if int(item["id"]) == target_id), None)
            if not record:
                continue
            target_name = record.get("api_name") or record.get("name") or str(target_id)
            target_query = f'{target_name} {section or ""}'.strip()
            complementary = retrieve(target_query, top_k=len(documents))
            complementary = [doc for doc in complementary if int(doc["pokemon_id"]) == target_id]
            if section == "encounters_strict":
                complementary = [doc for doc in complementary if doc["section"] == "encounters"]
            elif section in ("obtainment", "encounters"):
                complementary = [doc for doc in complementary if doc["section"] in ("obtainment", "encounters")]
            elif section:
                complementary = [doc for doc in complementary if doc["section"] == section]
            ranked.extend(complementary)
        unique = {}
        for doc in ranked:
            current = unique.get(doc["document_id"])
            if current is None or float(doc.get("score", 0.0)) > float(current.get("score", 0.0)):
                unique[doc["document_id"]] = doc
        ranked = list(unique.values())

    if not ranked:
        return [], "missing_target_section" if section else "no_relevant_documents"
    ranked = [doc for doc in ranked if float(doc.get('score', 0.0)) >= MIN_EVIDENCE_SCORE]
    if not ranked:
        return [], 'below_similarity_threshold'

    if targets:
        missing = set(targets) - {int(d["pokemon_id"]) for d in ranked}
        if missing:
            return [], "missing_comparison_evidence"

        # Primeiro reserva uma fonte para CADA Pokémon solicitado.
        # Em seguida, inclui as fontes adicionais com maior similaridade.
        prioritized = []
        chosen_ids = set()
        for target in targets:
            candidate = next((d for d in ranked if int(d["pokemon_id"]) == target), None)
            if candidate and candidate["document_id"] not in chosen_ids:
                prioritized.append(candidate)
                chosen_ids.add(candidate["document_id"])
        prioritized.extend(d for d in ranked if d["document_id"] not in chosen_ids)
        ranked = prioritized

    selected_rows = [dict(row, rank=idx + 1) for idx, row in enumerate(ranked[:max(int(top_k), len(targets))])]
    return selected_rows, "ready"

print("Recuperação TF-IDF: escopo, várias espécies e cobertura individual configurados.")


Recuperação TF-IDF: escopo, várias espécies e cobertura individual configurados.


## 5 · Montagem do contexto e contrato da resposta
O FLAN-T5 recebe somente a **pergunta**, **instruções de fundamentação** e os **trechos escolhidos**. O modelo produz texto livre; o Python mantém IDs, fontes e trechos enviados ao prompt. Tipos, estatísticas, evoluções e formas de obtenção são conferidos contra esses trechos; quando a geração omite fatos obrigatórios, a resposta é reparada em linguagem natural sem expor o documento bruto.

**Nota:** validação de citações e trechos é um controle mecânico, não um verificador semântico. A resposta pode continuar imprecisa mesmo após passar pelo controle.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, top_k=5):
    query_vec = vectorizer.transform([normalize_query(query)])
    similarities = cosine_similarity(query_vec, index_matrix).flatten()
    ranked_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in ranked_indices:
        score = similarities[idx]
        if score <= 0:
            continue
        doc = documents[idx].copy()
        doc['score'] = float(score)
        results.append(doc)
        if len(results) >= top_k:
            break
    return results

# Realiza o assert de sanidade com o build_context da célula 8 (que não redefine nada aqui)
ctx, chosen = build_context(select_evidence('Qual é a evolução de Eevee?', top_k=5)[0])
assert ctx and any('evolution' in d['document_id'] for d in chosen)
print('Recuperador e teste de sanidade prontos utilizando definições limpas da célula 8.')

Recuperador e teste de sanidade prontos utilizando definições limpas da célula 8.


## 6 · FLAN-T5 local como gerador


In [7]:
import importlib.util
import os
import subprocess
import sys
import time

FLAN_MODEL_NAME = os.environ.get('FLAN_MODEL_NAME', 'google/flan-t5-small')
PROVISION_FLAN_MODEL = os.environ.get(
    'PROVISION_FLAN_MODEL', '1' if IN_COLAB else '0') == '1'
PRELOAD_FLAN_MODEL = os.environ.get(
    'PRELOAD_FLAN_MODEL', '1' if IN_COLAB else '0') == '1'
DEVICE = 'cpu'
FLAN_TOKENIZER = None
FLAN_MODEL = None

if IN_COLAB:
    missing_generation_packages = [
        package for package in ('transformers', 'sentencepiece', 'ipywidgets')
        if importlib.util.find_spec(package) is None
    ]
    if missing_generation_packages:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers>=4.44,<5', 'sentencepiece>=0.2', 'ipywidgets>=8',
        ])

if PROVISION_FLAN_MODEL:
    from huggingface_hub import snapshot_download
    snapshot_download(FLAN_MODEL_NAME)

def load_flan_t5():
    global FLAN_TOKENIZER, FLAN_MODEL, DEVICE
    if FLAN_TOKENIZER is not None and FLAN_MODEL is not None:
        return FLAN_TOKENIZER, FLAN_MODEL
    try:
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    except ImportError as error:
        raise RuntimeError('Instale transformers, sentencepiece e torch para usar FLAN-T5.') from error
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    FLAN_TOKENIZER = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME, local_files_only=True)
    FLAN_MODEL = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME, local_files_only=True).to(DEVICE).eval()
    return FLAN_TOKENIZER, FLAN_MODEL

def generate_prompt_with_flan_t5(prompt):
    tokenizer, model = load_flan_t5()
    input_tokens = token_count(tokenizer, prompt)
    if input_tokens > MAX_INPUT_TOKENS:
        raise RuntimeError(f'Prompt excede {MAX_INPUT_TOKENS} tokens: {input_tokens}.')
    inputs = tokenizer(prompt, return_tensors='pt', truncation=False).to(DEVICE)
    started = time.perf_counter()
    import torch
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False, num_beams=1)
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return {'answer': answer, 'model': FLAN_MODEL_NAME, 'input_tokens': input_tokens,
            'output_tokens': int(output_ids.shape[-1]),
            'latency_seconds': time.perf_counter() - started,
            'generation_config': {'max_new_tokens': 64, 'do_sample': False, 'num_beams': 1}}

def generate_with_flan_t5(question, context):
    return generate_prompt_with_flan_t5(prompt_text(question, context))

def model_without_corpus(question):
    return generate_prompt_with_flan_t5(f'Responda em português de forma breve: {question}')

if PRELOAD_FLAN_MODEL:
    os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
    try:
        from transformers.utils import logging as transformers_logging
        transformers_logging.set_verbosity_error()
    except ImportError:
        pass

if PRELOAD_FLAN_MODEL:
    load_flan_t5()
    print('FLAN-T5 carregado e pronto para o chat.')

model_state = 'pré-carregado' if FLAN_MODEL is not None else 'sob demanda'
print('Gerador local:', FLAN_MODEL_NAME, f'| pesos {model_state}; dispositivo:', DEVICE)


Gerador local: google/flan-t5-small | pesos carregados sob demanda; dispositivo: cpu


## 7 · Validação mecânica de fontes e política de abstenção
Rejeitamos respostas se houver IDs inexistentes, fontes que não foram enviadas, citações não literais, ou fatos sem nenhuma evidência citada. **O sistema não deve tratar um ID citado como prova automática de correção factual.** Esse controle reduz respostas sem procedência, mas não elimina alucinações.

In [8]:
def evolution_transitions(rows):
    transitions = []
    pattern = r'-\s*([a-z0-9-]+)\s*->\s*([a-z0-9-]+)\.\s*(?:Conditions:\s*\((.*?)\))?'
    for row in rows:
        if row.get('section') != 'evolution':
            continue
        for match in re.finditer(pattern, str(row.get('content', '')), re.I | re.S):
            origin, target, conditions = match.groups()
            conditions = conditions or ''
            item_match = re.search(r'\bitem:\s*([^;.)]+)', conditions, re.I)
            level_match = re.search(r'\bminimum level:\s*(\d+)', conditions, re.I)
            transitions.append({
                'origin': origin.replace('-', ' ').title(),
                'target': target.replace('-', ' ').title(),
                'item': item_match.group(1).strip().replace('-', ' ').title() if item_match else '',
                'level': level_match.group(1) if level_match else ''})
    return transitions

def evolution_answer_is_complete(answer, rows):
    normalized_answer = normalize_plain(answer)
    transitions = evolution_transitions(rows)
    if not transitions:
        return True
    return all(
        normalize_plain(step['target']) in normalized_answer
        and (not step['level'] or step['level'] in normalized_answer)
        and (not step['item'] or normalize_plain(step['item']) in normalized_answer)
        for step in transitions)

def non_evolving_answer(rows, question):
    if not re.search(r'\b(?:evolu\w*|evolv\w*)\b', normalize_plain(question)):
        return None
    names = [str(row.get('pokemon_name', 'Pokémon')) for row in rows
             if row.get('section') == 'evolution' and re.search(r'does not evolve', str(row.get('content', '')), re.I)]
    if names:
        return '; '.join(f'{name} não evolui para outro Pokémon' for name in dict.fromkeys(names)) + '.'
    return None

def requested_stat_name(question):
    normalized = normalize_plain(question)
    patterns = (
        ('special attack', r'\b(?:ataque especial|special attack)\b'),
        ('special defense', r'\b(?:defesa especial|special defense)\b'),
        ('speed', r'\b(?:velocidade|speed)\b'),
        ('attack', r'\b(?:ataqu\w*|attack\w*)\b'),
        ('defense', r'\b(?:defes\w*|defens\w*)\b'),
        ('hp', r'\b(?:hp|pontos de vida|vida)\b'))
    return next((name for name, pattern in patterns if re.search(pattern, normalized)), None)

def profile_types_pt(row):
    content = str(row.get('content', ''))
    match = re.search(r'Types PT-BR:\s*([^.]*)\.', content, re.I)
    if not match:
        match = re.search(r'Types in Generation III:\s*([^.]*)\.', content, re.I)
    translations = {'fire':'fogo','water':'água','grass':'planta','electric':'elétrico','flying':'voador','poison':'venenoso','ground':'terrestre','rock':'pedra','bug':'inseto','ghost':'fantasma','steel':'aço','fighting':'lutador','psychic':'psíquico','ice':'gelo','dragon':'dragão','dark':'sombrio','normal':'normal'}
    return [translations.get(value.strip().casefold(), value.strip()) for value in match.group(1).split(',') if value.strip()] if match else []

def is_acquisition_question(question):
    normalized = normalize_plain(question)
    return bool(re.search(r'\b(?:onde|where|encontr\w*|localiz\w*|captur\w*|peg\w*|find|catch|obter|obten\w*|conseguir|receber|ganhar|obtain|gift|get)\b', normalized))

def acquisition_facts(rows):
    facts, seen = [], set()
    pattern = re.compile(r'^-\s*([^;\n]+);\s*method:\s*([^;\n]+);\s*levels:\s*(\d+)-(\d+);', re.I | re.M)
    for row in rows:
        if row.get('section') not in {'encounters', 'obtainment'}:
            continue
        for match in pattern.finditer(str(row.get('content', ''))):
            location, method, minimum, maximum = (value.strip() for value in match.groups())
            fact = {'pokemon': str(row.get('pokemon_name', 'Pokémon')).strip(),
                    'location': location.replace('-', ' ').title(),
                    'method': normalize_plain(method),
                    'minimum_level': minimum, 'maximum_level': maximum}
            key = tuple(fact.values())
            if key not in seen:
                seen.add(key); facts.append(fact)
    return facts

def acquisition_answer(rows, question):
    if not is_acquisition_question(question):
        return None
    facts = acquisition_facts(rows)
    methods_by_pokemon = {}
    for row in rows:
        match = re.search(r'Documented mechanisms:\s*([^.]*)\.', str(row.get('content', '')), re.I)
        if match:
            methods_by_pokemon.setdefault(str(row.get('pokemon_name', 'Pokémon')), set()).update(
                normalize_plain(value) for value in re.split(r'\s*,\s*|\s+and\s+', match.group(1)) if value.strip())
    if not facts and not methods_by_pokemon:
        return None
    facts_by_pokemon = {}
    for fact in facts:
        facts_by_pokemon.setdefault(fact['pokemon'], []).append(fact)
    pokemon_names = list(dict.fromkeys(
        [str(row.get('pokemon_name', 'Pokémon')) for row in rows] + [fact['pokemon'] for fact in facts]))
    sentences = []
    for name in pokemon_names:
        pokemon_facts = facts_by_pokemon.get(name, [])
        if pokemon_facts:
            for fact in pokemon_facts[:6]:
                location, method = fact['location'], fact['method']
                minimum, maximum = fact['minimum_level'], fact['maximum_level']
                level = f'no nível {minimum}' if minimum == maximum else f'entre os níveis {minimum} e {maximum}'
                if method == 'gift':
                    sentence = f'{name} é obtido como presente em {location}, {level}'
                else:
                    method_text = {'walk': 'caminhando', 'surf': 'usando Surf',
                                   'old rod': 'pescando com a Old Rod', 'good rod': 'pescando com a Good Rod',
                                   'super rod': 'pescando com a Super Rod', 'rock smash': 'usando Rock Smash'}.get(method, f'pelo método {method}')
                    sentence = f'{name} pode ser encontrado em {location}, {method_text}, {level}'
                sentences.append(sentence + '.')
            continue
        methods = methods_by_pokemon.get(name, set())
        if 'evolution' in methods:
            sentences.append(f'{name} é registrado no corpus como obtido por evolução e não possui local de encontro selvagem documentado.')
        elif methods:
            sentences.append(f'O corpus não informa um local de encontro para {name}.')
    if not sentences:
        return None
    return 'No Pokémon FireRed, ' + ' '.join(sentences)

def is_profile_summary_question(question):
    return bool(re.search(
        r'\b(?:fale|sabe|sabre|conte|descreva|apresente|resuma|quem e|tell me about|who is)\b',
        normalize_plain(question)))

def profile_answer(rows, question):
    if not is_profile_summary_question(question):
        return None
    row = next((item for item in rows if item.get('section') == 'profile'), None)
    if not row:
        return None
    content = str(row.get('content', ''))
    name = str(row.get('pokemon_name', 'Pokémon'))
    dex_match = re.search(r'National Pokédex\s*#(\d+)', content, re.I)
    types = profile_types_pt(row)
    opening = name
    if dex_match:
        opening += f' é o Pokémon #{dex_match.group(1)}'
    if types:
        connector = ',' if dex_match else ' é'
        opening += f"{connector} dos tipos {' e '.join(types)}"
    sentences = [opening + '.']
    editorial_description = EDITORIAL_DESCRIPTION_BY_ID.get(int(row.get('pokemon_id', 0)))
    description_match = re.search(
        r'Descri[cç][aã]o:\s*(.+?)(?=\r?\n(?:Habitat|Height|Capture rate|Pok[eé]mon|Types(?: in| PT-BR)?):|\Z)',
        content, re.I | re.S)
    if editorial_description:
        description = editorial_description.strip().rstrip(' .')
    elif description_match:
        description = re.sub(r'\s+', ' ', description_match.group(1)).strip().rstrip(' .')
    else:
        description = ''
    if description:
        sentences.append(f'{description}.')
    measures = re.search(
        r'Height:\s*(\d+(?:\.\d+)?)\s*m;\s*weight:\s*(\d+(?:\.\d+)?)\s*kg', content, re.I)
    habitat_match = re.search(r'Habitat:\s*([^.]*)\.', content, re.I)
    details = []
    if measures:
        height = f'{float(measures.group(1)):g}'.replace('.', ',')
        weight = f'{float(measures.group(2)):g}'.replace('.', ',')
        details.extend([f'Mede {height} m', f'pesa {weight} kg'])
    if habitat_match:
        habitat = normalize_plain(habitat_match.group(1))
        habitat_text = {'waters edge': 'vive próximo à água', 'forest': 'vive em florestas',
                        'mountain': 'vive em regiões montanhosas', 'grassland': 'vive em campos',
                        'cave': 'vive em cavernas', 'urban': 'vive em áreas urbanas', 'rare': 'é uma espécie rara'}.get(
                            habitat, f"vive em habitat {habitat.replace('-', ' ')}")
        details.append(habitat_text)
    if details:
        sentences.append(', '.join(details[:-1]) + (' e ' if len(details) > 1 else '') + details[-1] + '.')
    return ' '.join(sentences)

def type_matchup_answer(rows, question):
    row = next((item for item in rows if item.get('section') == 'type_matchups'), None)
    if not row: return None
    normalized_question = normalize_plain(question)
    names = {'rock':'pedra','electric':'elétrico','water':'água','fighting':'lutador','fire':'fogo','steel':'aço','bug':'inseto','grass':'planta','ground':'terrestre'}
    weak, resistant, immune = [], [], []
    for line in str(row.get('content', '')).splitlines():
        match = re.match(r'-\s*(4x|2x) weak:\s*([^.]*)', line, re.I)
        if match: weak.append((match.group(2), match.group(1))); continue
        match = re.match(r'-\s*(0\.5x|0\.25x) resistant:\s*([^.]*)', line, re.I)
        if match: resistant.append((match.group(2), match.group(1))); continue
        match = re.match(r'-\s*immune \(0x\):\s*([^.]*)', line, re.I)
        if match: immune.append(match.group(1))
    def render(groups):
        rendered = []
        for values, multiplier in groups:
            translated = [names.get(normalize_plain(value), value.strip()) for value in values.split(',')]
            joined = ', '.join(translated[:-1]) + (' e ' if len(translated) > 1 else '') + translated[-1]
            rendered.append(f"{joined} ({multiplier.replace('x', '×')})")
        return ', '.join(rendered)
    pokemon = row.get('pokemon_name', 'O Pokémon')
    if re.search(r'\b(?:fraquez|weak)\w*\b', normalized_question) and weak:
        return f"{pokemon} é fraco contra {render(weak)}."
    if re.search(r'\b(?:resist)\w*\b', normalized_question) and resistant:
        return f"{pokemon} resiste a {render(resistant)}."
    if re.search(r'\b(?:imun)\w*\b', normalized_question) and immune:
        translated = [names.get(normalize_plain(value), value.strip()) for value in immune[0].split(',')]
        return f"{pokemon} é imune a {' e '.join(translated)}."
    return None

def structured_answer(rows, question):
    acquisition = acquisition_answer(rows, question)
    if acquisition:
        return acquisition
    matchup = type_matchup_answer(rows, question)
    if matchup:
        return matchup
    if is_type_listing_question(question):
        answers = [f"{row['pokemon_name']} é do tipo {' e '.join(profile_types_pt(row))}"
                   for row in rows if profile_types_pt(row)]
        if answers:
            return '; '.join(answers) + '.'
    profile = profile_answer(rows, question)
    if profile:
        return profile
    transitions = evolution_transitions(rows)
    if transitions:
        portuguese = bool(re.search(r'\b(como|quais|qual|evolu)\b', normalize_plain(question)))
        steps = []
        for step in transitions:
            if portuguese:
                condition = f"usando {step['item']}" if step['item'] else f"no nível {step['level']}" if step['level'] else ''
                steps.append(f"{step['origin']} evolui para {step['target']} {condition}".strip())
            else:
                condition = f"using {step['item']}" if step['item'] else f"at level {step['level']}" if step['level'] else ''
                steps.append(f"{step['origin']} evolves into {step['target']} {condition}".strip())
        return '; '.join(steps) + '.'
    non_evolving = non_evolving_answer(rows, question)
    if non_evolving:
        return non_evolving
    normalized_question = normalize_plain(question)
    if is_ranking_question(question):
        stat = requested_stat_name(question)
        label = {'speed':'velocidade','attack':'ataque','defense':'defesa','special attack':'ataque especial','special defense':'defesa especial','hp':'HP'}[stat]
        ranked = []
        for row in rows:
            match = re.search(rf'-\s*{re.escape(stat)}:\s*(\d+(?:\.\d+)?)', str(row.get('content', '')), re.I)
            if match: ranked.append((float(match.group(1)), row['pokemon_name'], match.group(1)))
        ranked.sort(key=lambda item: (-item[0], item[1]))
        if ranked:
            limit = ranking_limit(question)
            shown = ranked if limit is None else ranked[:limit]
            prefix = 'Os maiores valores são: ' if limit is None else f'Os {len(shown)} maiores valores são: '
            return prefix + '; '.join(f'{name}: {label} {value}' for _, name, value in shown) + '.'
    if re.search(r'\b(?:tipos?|types?)\b', normalized_question) or is_type_listing_question(question):
        if re.search(r'\b(?:mesm[oa]s?|igual(?:es)?|same)\b', normalized_question) and len(rows) > 1:
            names = ', '.join(row['pokemon_name'] for row in rows[1:])
            types = ' e '.join(profile_types_pt(rows[0]))
            return f"{names} têm os mesmos tipos ({types}) que {rows[0]['pokemon_name']}."
        answers = [f"{row['pokemon_name']} é do tipo {' e '.join(profile_types_pt(row))}"
                   for row in rows if profile_types_pt(row)]
        if answers:
            return '; '.join(answers) + '.'
    if re.search(r'\b(?:atribut\w*|atronut\w*|estatistic\w*|stats?)\b', normalized_question) and not requested_stat_name(question):
        labels = {'hp':'HP','attack':'ataque','defense':'defesa','special attack':'ataque especial','special defense':'defesa especial','speed':'velocidade'}
        summaries = []
        for row in rows:
            values = []
            for key, label in labels.items():
                match = re.search(rf'-\s*{re.escape(key)}:\s*(\d+(?:\.\d+)?)', str(row.get('content', '')), re.I)
                if match: values.append(f'{label} {match.group(1)}')
            if values: summaries.append(f"{row['pokemon_name']}: {', '.join(values)}")
        if summaries: return '; '.join(summaries) + '.'
    stat = requested_stat_name(question)
    if stat:
        labels = {'speed':'velocidade base', 'attack':'ataque base', 'defense':'defesa base',
                  'special attack':'ataque especial base', 'special defense':'defesa especial base', 'hp':'HP base'}
        values = []
        for row in rows:
            match = re.search(rf'-\s*{re.escape(stat)}:\s*(\d+(?:\.\d+)?)', str(row.get('content', '')), re.I)
            if match:
                values.append((row['pokemon_name'], match.group(1)))
        if values:
            answer = '; '.join(f"{name} tem {labels[stat]} {value}" for name, value in values)
            if len(values) == 2 and float(values[0][1]) != float(values[1][1]):
                greater = values[0][0] if float(values[0][1]) > float(values[1][1]) else values[1][0]
                answer += f'; {greater} possui o maior valor'
            return answer + '.'
    return None

def structured_answer_is_complete(answer, rows, question):
    expected = structured_answer(rows, question)
    if not expected:
        return True
    normalized_answer = normalize_plain(answer)
    if is_type_listing_question(question):
        return all(normalize_plain(row.get('pokemon_name', '')) in normalized_answer
                   for row in rows if row.get('section') == 'profile')
    acquisition = acquisition_facts(rows) if is_acquisition_question(question) else []
    if acquisition:
        for fact in acquisition:
            location_anchor = next((token for token in normalize_plain(fact['location']).split() if len(token) >= 4), '')
            if normalize_plain(fact['pokemon']) not in normalized_answer or (location_anchor and location_anchor not in normalized_answer):
                return False
            if fact['method'] == 'gift' and not re.search(r'\b(?:presente|gift|receb\w*|obt\w*)\b', normalized_answer):
                return False
        return True
    if evolution_transitions(rows):
        return evolution_answer_is_complete(answer, rows)
    stat = requested_stat_name(question)
    if stat:
        for row in rows:
            match = re.search(rf'-\s*{re.escape(stat)}:\s*(\d+(?:\.\d+)?)', str(row.get('content', '')), re.I)
            if match and (normalize_plain(row['pokemon_name']) not in normalized_answer or match.group(1) not in normalized_answer):
                return False
        return True
    return all(all(normalize_plain(value) in normalized_answer for value in profile_types_pt(row))
               for row in rows if profile_types_pt(row))

def retrieval_answer(rows, question):
    if not rows:
        return None
    structured = structured_answer(rows, question)
    if structured:
        return structured
    excerpts = [str(row.get('content', '')).strip() for row in rows[:3] if str(row.get('content', '')).strip()]
    return '\n\n'.join(excerpts)[:9000] if excerpts else None

def clean_public_answer(answer):
    return str(answer or '').strip()

SCOPE_LIMIT_REASONS = {'out_of_scope', 'outside_kanto_dex', 'needs_comparison_criterion'}

def answer_question(question, top_k=5, generate=False, generator=None, original_question=None):
    if not isinstance(question, str) or not question.strip():
        raise ValueError('Pergunta vazia.')
    user_question = original_question or question
    canonical_question, corrections = canonicalize_question(question)
    retrieved, gate = select_evidence(canonical_question, top_k=top_k)
    language = detect_response_language(user_question) if 'detect_response_language' in globals() else 'pt'
    preflight_error = None
    if generate and generator is None:
        try:
            tokenizer, _ = load_flan_t5()
            context, used = build_context(retrieved, canonical_question, tokenizer)
        except Exception as error:
            preflight_error = error
            context, used = build_context(retrieved)
    else:
        context, used = build_context(retrieved)
    targets = explicitly_mentioned_ids(canonical_question)
    result = {'question': user_question, 'canonical_question': canonical_question,
              'normalized_query': normalize_query(canonical_question),
              'spelling_corrections': corrections, 'language': language,
              'retriever': 'tfidf_cosine_from_notebook_02',
              'generator': FLAN_MODEL_NAME if generate else None,
              'status': 'retrieval_only' if used else (
                  'out_of_scope' if gate in SCOPE_LIMIT_REASONS else 'insufficient_evidence'),
              'gate_reason': gate, 'answer': None if used else REFUSAL,
              'answer_mode': 'extractive',
              'source_documents': [{key: row.get(key) for key in ('document_id','pokemon_id','pokemon_name','section','score','factual_scope','source_file')} for row in used],
              'supporting_excerpts': [],
              'context_chars': len(context),
              'documents_in_prompt': [row['document_id'] for row in used]}
    collection_answer = structured_answer(retrieved, user_question) if (is_type_listing_question(user_question) or is_ranking_question(user_question)) else None
    if collection_answer:
        return {**result, 'status': 'answered', 'answer': collection_answer, 'answer_mode': 'grounded_structured',
                'source_documents': [{key: row.get(key) for key in ('document_id','pokemon_id','pokemon_name','section','score','factual_scope','source_file')} for row in retrieved],
                'supporting_excerpts': [{'document_id': row['document_id'], 'quote': row['content']} for row in retrieved]}
    if gate == 'needs_comparison_criterion':
        comparison_names = pokemon_names_by_ids(targets)
        subjects = ' e '.join(comparison_names) if comparison_names else 'esses Pokémon'
        return {**result, 'status': 'clarification_needed',
                'answer': (f'Em qual atributo você quer comparar {subjects}: ataque, defesa, '
                           'velocidade ou HP?'),
                'answer_mode': 'clarification', 'source_documents': [],
                'supporting_excerpts': []}
    if preflight_error is not None:
        fallback_answer = structured_answer(used, user_question) or retrieval_answer(used, user_question)
        return {**result, 'status': 'provider_unavailable', 'answer': fallback_answer,
                'answer_mode': 'extractive', 'generation_error': type(preflight_error).__name__}
    if not used or not generate:
        return result if not used else {**result, 'answer': retrieval_answer(used, user_question)}
    if len(targets) > 1 and not set(targets).issubset({int(row['pokemon_id']) for row in used}):
        return {**result, 'status': 'insufficient_evidence', 'answer': REFUSAL}
    provider = generator or generate_with_flan_t5
    try:
        payload = provider(user_question, context)
        answer = payload.get('answer', '') if isinstance(payload, dict) else str(payload)
    except Exception as error:
        return {**result, 'status': 'provider_unavailable', 'answer': retrieval_answer(used, user_question),
                'generation_error': type(error).__name__}
    metadata = {key: value for key, value in (payload if isinstance(payload, dict) else {}).items() if key != 'answer'}
    metadata['documents_in_prompt'] = [row['document_id'] for row in used]
    structured = structured_answer(used, user_question)
    if structured and (is_acquisition_question(user_question) or is_profile_summary_question(user_question)
                       or requested_section(user_question) == 'type_matchups'
                       or non_evolving_answer(used, user_question)
                       or is_type_listing_question(user_question) or evolution_transitions(used)):
        metadata.update(response_mode='grounded_structured', raw_answer=answer)
        answer = structured
    elif structured and not structured_answer_is_complete(answer, used, user_question):
        fallback = retrieval_answer(used, user_question)
        if fallback:
            reason = ('missing_evolution_conditions' if not is_acquisition_question(user_question) and evolution_transitions(used)
                      else 'missing_structured_facts')
            metadata.update(response_mode='grounded_repair', fallback_reason=reason)
            return {**result, 'status': 'answered', 'answer': structured or fallback, 'answer_mode': 'generative',
                    'supporting_excerpts': [{'document_id': row['document_id'], 'quote': row['content']} for row in used],
                    'generation_metadata': metadata}
    if (structured and requested_stat_name(user_question)
            and len({int(row['pokemon_id']) for row in used}) == 1):
        metadata.update(response_mode='focused_structured', raw_answer=answer)
        answer = structured
    if not answer.strip() or answer.strip().casefold() in {'não sei', 'nao sei'}:
        return {**result, 'status': 'insufficient_evidence', 'answer': REFUSAL}
    return {**result, 'status': 'answered', 'answer': clean_public_answer(answer),
            'answer_mode': 'generative',
            'supporting_excerpts': [{'document_id': row['document_id'], 'quote': row['content']} for row in used],
            'generation_metadata': metadata}


## 8 · Testes do contrato e fontes selecionadas


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


In [9]:
CHATBOT_RETRIEVAL_METRICS = EVAL_DIR / 'chatbot_retrieval_metrics_by_query.csv'
CHATBOT_RETRIEVAL_SUMMARY = EVAL_DIR / 'chatbot_retrieval_metrics_summary.csv'
CHATBOT_RETRIEVAL_RANKINGS = EVAL_DIR / 'chatbot_retrieval_rankings.csv'
EVAL_KS = (1, 3, 5, 10)
queries_eval = pd.read_csv(EVAL_DIR / 'evaluation_queries.csv', dtype={'query_id': str})
qrels_eval = pd.read_csv(EVAL_DIR / 'evaluation_qrels.csv', dtype={'query_id': str, 'document_id': str})
qrels_by_query_eval = {qid: set(group.loc[group.relevance > 0, 'document_id']) for qid, group in qrels_eval.groupby('query_id')}

def retrieval_metrics(ranked_ids, relevant, k):
    hits = [int(document_id in relevant) for document_id in ranked_ids[:k]]
    return {'hit_at_k': float(any(hits)), 'precision_at_k': sum(hits) / k,
            'recall_at_k': sum(hits) / len(relevant) if relevant else 0.0,
            'mrr_at_k': next((1 / (index + 1) for index, hit in enumerate(hits) if hit), 0.0)}

rows, rankings = [], []
for query in queries_eval.itertuples(index=False):
    relevant = qrels_by_query_eval.get(query.query_id, set())
    if not relevant:
        continue
    raw = retrieve(query.question, top_k=max(EVAL_KS))
    filtered, gate_reason = select_evidence(query.question, top_k=max(EVAL_KS))
    label_set = 'reviewed' if getattr(query, 'review_status', '') == 'reviewed' else 'provisional'
    for method, result in (('tfidf_raw', raw), ('tfidf_filtered', filtered)):
        ranking = [item['document_id'] for item in result]
        rankings.extend({'query_id': query.query_id, 'method': method, 'rank': rank,
                         'document_id': item['document_id'], 'section': item.get('section'),
                         'pokemon_id': item.get('pokemon_id'), 'gate_reason': gate_reason,
                         'label_set': label_set} for rank, item in enumerate(result, 1))
        for k in EVAL_KS:
            rows.append({'query_id': query.query_id, 'method': method, 'k': k,
                         'label_set': label_set, 'gate_reason': gate_reason,
                         'species_coverage': len({item['pokemon_id'] for item in result[:k]}),
                         **retrieval_metrics(ranking, relevant, k)})
chatbot_metrics = pd.DataFrame(rows)
chatbot_rankings = pd.DataFrame(rankings)
chatbot_summary = chatbot_metrics.groupby(['label_set', 'method', 'k'], as_index=False)[
    ['hit_at_k', 'precision_at_k', 'recall_at_k', 'mrr_at_k', 'species_coverage']].mean()
chatbot_metrics.to_csv(CHATBOT_RETRIEVAL_METRICS, index=False, encoding='utf-8')
chatbot_rankings.to_csv(CHATBOT_RETRIEVAL_RANKINGS, index=False, encoding='utf-8')
chatbot_summary.to_csv(CHATBOT_RETRIEVAL_SUMMARY, index=False, encoding='utf-8')
print('Comparação exploratória tfidf_raw × tfidf_filtered:', CHATBOT_RETRIEVAL_SUMMARY)


Comparação exploratória tfidf_raw × tfidf_filtered: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\evaluation\chatbot_retrieval_metrics_summary.csv


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


In [12]:
# O FLAN-T5 retorna texto livre; as fontes continuam selecionadas pelo Python.
target = "Quais são os tipos do Charizard?"
rows, gate = select_evidence(target, top_k=5)
context, used = build_context(rows)
assert gate == "ready" and used
assert used[0]["document_id"] in context
assert token_count is not None and "Resposta:" in prompt_text(target, context)
stat_content = 'Base stats for Charmander:\n- attack: 52.\n- defense: 43.\n- hp: 39.\n- speed: 65.'
assert rank_passages('Qual seu ataque base?', stat_content) == ['- attack: 52.']
focused = answer_question(
    'Qual o ataque base do Charmander?', generate=True,
    generator=lambda question, prompt: {
        'answer': 'Charmander stats: attack 52, defense 43, hp 39, speed 65.',
        'model': 'contract-double',
    },
)
assert focused['answer'] == 'Charmander tem ataque base 52.'
assert focused['answer_mode'] == 'generative'
assert focused['generation_metadata']['response_mode'] == 'focused_structured'
location_rows, location_gate = select_evidence('Onde encontro Pikachu e Dragonite?', top_k=5)
assert location_gate == 'ready'
assert '025_encounters_01' in {row['document_id'] for row in location_rows}
grounded_location = answer_question(
    'Onde encontro Pikachu e Dragonite?', generate=True,
    generator=lambda question, prompt: {'answer': 'Pikachu e Dragonite.', 'model': 'contract-double'},
)
assert 'Pikachu pode ser encontrado' in grounded_location['answer']
assert 'Dragonite' in grounded_location['answer'] and 'obtido por evolução' in grounded_location['answer']
assert grounded_location['answer_mode'] == 'generative'
assert grounded_location['generation_metadata']['response_mode'] == 'grounded_structured'
profile_summary = answer_question(
    'Fale um pouco sobre Dragonite', generate=True,
    generator=lambda question, prompt: {'answer': 'Dragonite is a dragon and flying Pokémon.', 'model': 'contract-double'},
)
assert profile_summary['answer'].startswith('Dragonite é o Pokémon #149')
assert 'vive próximo à água' in profile_summary['answer']
assert profile_summary['answer_mode'] == 'generative'
profile_typo = answer_question(
    'O que sabre sobre o Charizard?', generate=True,
    generator=lambda question, prompt: {'answer': 'Charizard (National Pokédex #006).', 'model': 'contract-double'},
)
assert 'tipos fogo e voador' in profile_typo['answer']
assert 'Mede 1,7 m' in profile_typo['answer']
profile_overview = answer_question(
    'O que sabe sobre o Charizard?', generate=True,
    generator=lambda question, prompt: {'answer': 'Pokémon: Charizard (National Pokédex #006)', 'model': 'contract-double'},
)
assert 'tipos fogo e voador' in profile_overview['answer']
assert 'Mede 1,7 m' in profile_overview['answer']
assert 'Geração III' not in profile_overview['answer']
assert 'Descrição da Pokédex' not in profile_overview['answer']
print("Seleção de fontes, contexto focado e contrato local: OK.")


Seleção de fontes, contexto focado e contrato local: OK.


## 9 · Demonstração do RAG com FLAN-T5


In [13]:
DEMO_QUESTION = 'Qual é a velocidade-base do Pikachu?'
RUN_LIVE_DEMO = False
if RUN_LIVE_DEMO:
    print(answer_question(DEMO_QUESTION, generate=True))
else:
    print('Demonstração FLAN-T5 desativada. Provisione o cache e altere RUN_LIVE_DEMO=True para carregar o modelo.')


Demonstração FLAN-T5 desativada. Provisione o cache e altere RUN_LIVE_DEMO=True para carregar o modelo.


## 10 · Conversa contextual com memória curta
Uma sessão guarda os Pokémon mencionados (inclusive em perguntas com duas espécies), reformula seguimentos antes do TF-IDF e solicita esclarecimento quando um pronome singular é ambíguo. Não usa respostas passadas como fonte factual.


In [14]:
class PokedexChatSession:
    """Memória curta de referências; fatos sempre vêm do TF-IDF + documentos."""
    def __init__(self):
        self.active_ids = []
        self.last_section = None
        self.last_question = None
        self.pending_question = None
        self.pending_candidates = []
        self.pending_kind = None
        self.turns = []

    def _is_followup(self, question):
        if not self.active_ids or explicitly_mentioned_ids(question):
            return False
        q = normalize_plain(question)
        if len(q.split()) > 12:
            return False
        patterns = [
            r"\b(ele|ela|dele|dela|deles|delas|seu|sua|seus|suas|ambos|ambas)\b",
            r"\b(it|its|they|them|their|both)\b",
            r"^(?:e |and )?(?:qual|quais|what|which|how|where|when|como|onde|quando|de quem|para quem)\b",
            r"^(?:e |and )?(?:em que nivel|qual nivel|at what level)\b",
            r"^(?:e |and )?(?:o |a |os |as )?(?:tipo|tipos|fraqueza|evolucao|nivel|type|types|evolution|level)\b",
            r"^(?:ataque|defesa|velocidade|hp|attack|defense|speed)$"
        ]
        return any(re.search(p, q) for p in patterns)

    @staticmethod
    def _is_plural_reference(question):
        q = normalize_plain(question)
        return bool(re.search(r"\b(eles|elas|deles|delas|ambos|ambas|dos dois|das duas|they|them|their|both|each)\b", q))

    @staticmethod
    def _is_singular_reference(question):
        q = normalize_plain(question)
        return bool(re.search(r"\b(dele|dela|ele|ela|seu|sua|it|its|him|her)\b", q))

    @staticmethod
    def _is_comparison_question(question):
        q = normalize_plain(question)
        return bool(re.search(r'\b(?:mais forte|melhor|stronger|better)\b', q))

    @staticmethod
    def _comparison_attribute(question):
        labels = {'attack': 'ataque', 'defense': 'defesa', 'speed': 'velocidade', 'hp': 'HP',
                  'special attack': 'ataque especial', 'special defense': 'defesa especial'}
        return labels.get(requested_stat_name(question))

    def _clarification(self, question):
        en = detect_response_language(question) == "en"
        names = pokemon_names_by_ids(self.active_ids)
        if len(names) == 2:
            options = f"{names[0]} or {names[1]}" if en else f"{names[0]} ou {names[1]}"
        else:
            options = ", ".join(names)
        message = (f"Which Pokémon do you mean: {options}?" if en
                   else f"De qual Pokémon você está falando: {options}?")
        self.pending_question = question
        self.pending_candidates = list(self.active_ids)
        self.pending_kind = 'reference'
        return {
            "question": question, "original_question": question,
            "resolved_question": None, "context_used": True,
            "active_pokemon": names, "language": "en" if en else "pt",
            "status": "clarification_needed", "gate_reason": "ambiguous_reference",
            "answer": message, "answer_mode": "extractive",
            "source_documents": [], "supporting_excerpts": []
        }

    def _resolve(self, question, target_ids):
        en = detect_response_language(question) == "en"
        names = pokemon_names_by_ids(target_ids)
        joined = " and ".join(names) if en else " e ".join(names)
        q = normalize_plain(question)
        section = requested_section(question)
        if section is None and re.search(r"\b(nivel|level|pedra|stone)\b", q):
            section = "evolution"
        if section is None:
            section = self.last_section or "profile"
        if section == "evolution":
            if self.last_section in {"encounters", "obtainment"} and re.search(r"\b(evolucao|evolution|evolui|evolve|nivel|level)\b", q):
                return (f"Which Pokémon evolves into {joined} and under what conditions?" if en
                        else f"De qual Pokémon {joined} evolui e em quais condições?"), section
            return (f"What is the evolution line of {joined}, including conditions and levels?" if en
                    else f"Qual é a linha evolutiva de {joined}, incluindo condições e níveis?"), section
        if section == "profile" and re.search(r"\b(tipo|tipos|type|types)\b", q):
            return (f"What are the types of {joined}?" if en else f"Quais são os tipos de {joined}?"), section
        # Preserva as palavras do usuário, acrescentando o referente sem tags artificiais.
        return (f"{question.strip()} (about {joined})" if en else
                f"{question.strip()} (sobre {joined})"), section

    def _remember(self, question, target_ids, section, result):
        self.active_ids = list(target_ids)
        self.last_section = section
        self.last_question = question
        self.turns.append(result)
        self.turns = self.turns[-8:]

    def ask(self, question, generate=False, generator=None, top_k=5):
        if not isinstance(question, str) or not question.strip():
            raise ValueError("Pergunta vazia.")
        question = question.strip()
        explicit = explicitly_mentioned_ids(question)
        resolved = question
        context_used = False
        targets = list(explicit)

        # Resolve tanto uma espécie ambígua quanto o atributo de uma comparação.
        if self.pending_question is not None:
            attribute_label = self._comparison_attribute(question)
            if self.pending_kind == 'comparison' and attribute_label and not explicit:
                targets = list(self.pending_candidates)
                joined = ' e '.join(pokemon_names_by_ids(targets))
                resolved = f'Compare o {attribute_label} de {joined}.'
                section = 'stats'
                context_used = True
                self.pending_question = None
                self.pending_candidates = []
                self.pending_kind = None
            elif self.pending_kind == 'ranking' and attribute_label and not explicit:
                resolved = f'Quais são os pokemons com maior {attribute_label}?'
                section = 'stats'
                context_used = True
                self.pending_question = None
                self.pending_candidates = []
                self.pending_kind = None
            elif (self.pending_kind == 'reference' and len(explicit) == 1
                  and explicit[0] in self.pending_candidates
                  and len(normalize_plain(question).split()) <= 3):
                pending = self.pending_question
                self.pending_question = None
                self.pending_candidates = []
                self.pending_kind = None
                resolved, section = self._resolve(pending, explicit)
                context_used = True
                targets = list(explicit)
            else:
                self.pending_question = None
                self.pending_candidates = []
                self.pending_kind = None
                section = requested_section(question) or "profile"
        else:
            section = requested_section(question) or "profile"

        if not context_used:
            if explicit:
                if self.active_ids and self._is_singular_reference(question) and self._is_comparison_question(question):
                    targets = list(dict.fromkeys([*self.active_ids, *explicit]))
                    joined = ' e '.join(pokemon_names_by_ids(targets))
                    resolved = f'{question} (comparando {joined})'
                    context_used = True
                else:
                    targets = list(explicit)
            elif self._is_followup(question):
                if len(self.active_ids) > 1 and not self._is_plural_reference(question):
                    # Em contexto com duas espécies, "dele" e "qual evolução?" são ambíguos.
                    result = self._clarification(question)
                    self.turns.append(result)
                    self.turns = self.turns[-8:]
                    return result
                targets = list(self.active_ids)
                resolved, section = self._resolve(question, targets)
                context_used = True
            else:
                # Pergunta nova sem referência explícita: evita vazamento de contexto.
                targets = []

        result = answer_question(
            resolved, top_k=top_k, generate=generate,
            generator=generator, original_question=question
        )
        result["original_question"] = question
        result["resolved_question"] = resolved
        result["context_used"] = context_used
        result["active_pokemon"] = pokemon_names_by_ids(targets)
        if result.get('gate_reason') == 'needs_comparison_criterion':
            self.pending_question = resolved
            self.pending_candidates = list(targets)
            self.pending_kind = 'comparison' if targets else 'ranking'
            comparison_names = ' e '.join(pokemon_names_by_ids(targets))
            result['answer'] = (f'Em qual atributo você quer comparar {comparison_names}: '
                                'ataque, defesa, velocidade ou HP?')
            if not targets:
                result['answer'] = 'Em qual atributo você quer ranquear: ataque, defesa, velocidade ou HP?'
        self._remember(question, targets, section, result)
        return result


session = PokedexChatSession()
print("Sessão conversacional pronta: múltiplos Pokémon, pronomes e esclarecimentos.")


Sessão conversacional pronta: múltiplos Pokémon, pronomes e esclarecimentos.


### Testes de continuidade e ambiguidade (sem FLAN-T5)
Estes testes validam referências e recuperação. Execute-os após atualizar o índice do Notebook 02.


In [15]:
# Contratos de seleção independentes da ordem de definição das células.
assert requested_section("Como encontrar o Eevee?") == "encounters"
assert requested_section("Como evoluir o Charizard?") == "evolution"
rows, gate = select_evidence("Quais os pokemons com o mesmo tipo do charizard?", top_k=5)
assert gate == "ready" and {6, 146}.issubset({int(row["pokemon_id"]) for row in rows})
comparison_session = PokedexChatSession()
comparison_session.ask('Me fale sobre o Eevee', generate=False)
comparison_clarification = comparison_session.ask('Ele é mais forte que o Pikachu?', generate=False)
assert comparison_clarification['active_pokemon'] == ['Eevee', 'Pikachu']
assert 'Eevee e Pikachu' in comparison_clarification['answer']
comparison_answer = comparison_session.ask('Ataque', generate=False)
assert 'Eevee tem ataque base 55' in comparison_answer['answer']
assert 'Pikachu tem ataque base 55' in comparison_answer['answer']
print("Contratos de intenção, fontes e comparação por tipo: OK.")


Contratos de intenção, fontes e comparação por tipo: OK.


## 11 · Interface de chat opcional no Colab
Execute `open_chat_ui()` para abrir uma interface conversacional. O modelo é carregado uma única vez ao abrir o chat; a interface mostra somente as mensagens do usuário e as respostas finais do assistente. Fontes, modos e metadados permanecem no resultado Python para auditoria, mas não aparecem na conversa.

In [16]:
def chat_bubble(label, text, *, user=False):
    safe_text = html.escape(str(text or '')).replace('\n', '<br>')
    side = 'margin-left:auto' if user else 'margin-right:auto'
    background = '#dbeafe' if user else '#ffffff'
    return (
        f"<div style='max-width:78%;{side};margin-top:10px;margin-bottom:10px'>"
        f"<div style='font-size:12px;color:#64748b;margin:0 8px 4px'>{html.escape(label)}</div>"
        f"<div style='background:{background};border:1px solid #e2e8f0;border-radius:16px;"
        f"padding:10px 14px;color:#0f172a;line-height:1.5'>{safe_text}</div></div>")

def show_answer(result, question=None):
    if question:
        display(HTML(chat_bubble('Você', question, user=True)))
    answer = str(result.get('answer') or 'Não consegui formular uma resposta.').strip()
    display(HTML(chat_bubble('Smart Pokédex', answer)))
    return result

def open_chat_ui():
    try:
        import ipywidgets as widgets
    except ImportError:
        print('Instale ipywidgets para abrir a interface.')
        return None
    os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
    try:
        from transformers.utils import logging as transformers_logging
        transformers_logging.set_verbosity_error()
    except ImportError:
        pass
    load_flan_t5()
    conversation = PokedexChatSession()
    title = widgets.HTML("<h3 style='margin:0'>Smart Pokédex</h3><span style='color:#64748b'>Converse sobre os Pokémon de Kanto em FireRed.</span>")
    field = widgets.Text(placeholder='Digite sua pergunta...', layout=widgets.Layout(flex='1 1 auto'))
    action = widgets.Button(description='Enviar', button_style='primary', icon='paper-plane')
    transcript = widgets.HTML(value='', layout=widgets.Layout(
        border='1px solid #e2e8f0', padding='12px', height='420px', width='100%'))
    transcript_messages = []
    def append_message(label, text, *, user=False):
        transcript_messages.append(chat_bubble(label, text, user=user))
        transcript.value = (
            "<div style='height:390px;overflow-y:auto;overflow-x:hidden;display:flex;"
            "flex-direction:column;padding-right:8px'>" + ''.join(transcript_messages) + '</div>')
    status = widgets.HTML('')
    def run(_):
        prompt = field.value.strip()
        if not prompt: return
        field.value = ''
        action.disabled = True
        field.disabled = True
        status.value = "<span style='color:#64748b'>Pensando...</span>"
        append_message('Você', prompt, user=True)
        try:
            result = conversation.ask(prompt, generate=True)
            append_message('Smart Pokédex', result.get('answer'))
        except Exception as error:
            append_message('Smart Pokédex', f'Falha na geração: {type(error).__name__}: {error}')
        finally:
            action.disabled = False
            field.disabled = False
            status.value = ''
    action.on_click(run)
    display(widgets.VBox([title, transcript, widgets.HBox([field, action]), status],
                         layout=widgets.Layout(width='100%', max_width='850px')))
    return conversation

if IN_COLAB:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
CHAT_UI = open_chat_ui() if IN_COLAB else None


## 12 · Avaliação do gerador (separada da recuperação)
O Notebook 02 já mediu Precision@K, Recall@K e MRR. Aqui registramos **abstenção, cobertura de citações, integridade das evidências** e uma **planilha de revisão humana** sobre correção factual, suporte semântico e completude.

- As perguntas são apenas um conjunto inicial: revise e amplie antes de reportar resultados definitivos.
- **A avaliação com FLAN-T5 é opt-in** porque executa inferências locais e pode exigir recursos computacionais adicionais.
- Não calculamos automaticamente *fidelidade factual*: verificar substring não é provar que a afirmação está correta.
- Os JSONLs exportados registram respostas e IDs, sem credenciais ou segredos.

In [17]:
EVAL_QUESTIONS_PATH = EVAL_DIR / 'rag_generation_questions.csv'
EVAL_RESULTS_PATH = EVAL_DIR / 'rag_generation_results.jsonl'
EVAL_REVIEW_PATH = EVAL_DIR / 'rag_manual_review.csv'
seed_cases = [
    ('R01', 'Quais são os tipos do Charizard?', 'answerable', 'profile'),
    ('R02', 'Qual é a velocidade-base do Pikachu?', 'answerable', 'stats'),
    ('R03', 'Compare a velocidade-base de Charizard e Blastoise.', 'comparison', 'stats'),
    ('R04', 'Quais ginásios preciso vencer?', 'out_of_scope', 'none'),
    ('R05', 'Qual é o número do sapato do Pikachu?', 'missing_evidence', 'profile'),
]
if not EVAL_QUESTIONS_PATH.exists():
    pd.DataFrame(seed_cases, columns=['query_id','question','expected_case','expected_section']).to_csv(
        EVAL_QUESTIONS_PATH, index=False, encoding='utf-8')
def validate_generation_questions(frame, source_name):
    questions = frame.copy()
    questions.columns = [re.sub(r'\W+', '_', str(column).strip().casefold()).strip('_') for column in questions.columns]
    legacy_case_values = {'answerable', 'comparison', 'out_of_scope', 'missing_evidence'}
    if 'expected_case' not in questions.columns and 'category' in questions.columns:
        legacy_cases = questions['category'].map(
            lambda value: str(value).strip().casefold() if pd.notna(value) else ''
        )
        unknown_cases = sorted(set(legacy_cases) - legacy_case_values)
        if unknown_cases or legacy_cases.empty:
            raise ValueError(
                f'Esquema antigo em {source_name}: a coluna category não contém rótulos '
                f'de expected_case reconhecidos. Valores encontrados: {sorted(set(legacy_cases))}. '
                'Valores aceitos para migração: answerable, comparison, out_of_scope, '
                'missing_evidence. Não é seguro inferir a classe automaticamente.'
            )
        questions['expected_case'] = legacy_cases
    required = {'query_id', 'question', 'expected_case', 'expected_section'}
    missing = required - set(questions.columns)
    if missing:
        raise ValueError(f'Esquema nÃ£o suportado em {source_name}: colunas obrigatÃ³rias ausentes {sorted(missing)}; colunas encontradas {list(questions.columns)}. Nenhum campo foi convertido automaticamente.')
    if not questions['query_id'].is_unique:
        raise ValueError(f'{source_name}: query_id deve ser Ãºnico.')
    return questions

def load_generation_questions(path, seed_rows):
    raw = pd.read_csv(path)
    try:
        return validate_generation_questions(raw, path.name)
    except ValueError:
        normalized_columns = {re.sub(r'\W+', '_', str(column).strip().casefold()).strip('_') for column in raw.columns}
        thematic_legacy = 'expected_case' not in normalized_columns and 'category' in normalized_columns
        if not thematic_legacy:
            raise
        backup = path.with_name('rag_generation_questions_legacy_backup.csv')
        if not backup.exists():
            raw.to_csv(backup, index=False, encoding='utf-8')
        canonical = pd.DataFrame(
            seed_rows, columns=['query_id','question','expected_case','expected_section'])
        canonical.to_csv(path, index=False, encoding='utf-8')
        warnings.warn(
            f'CSV temático legado preservado em {backup.name}; {path.name} foi atualizado com o esquema canônico.',
            UserWarning,
        )
        return validate_generation_questions(canonical, path.name)

questions = load_generation_questions(EVAL_QUESTIONS_PATH, seed_cases)

def precheck_generation_case(row):
    selected, gate = select_evidence(row['question'], top_k=5)
    corpus_text = '\n'.join(str(document['content']).casefold() for document in documents)
    if row['expected_case'] in {'answerable', 'comparison'}:
        assert selected, f'{row["query_id"]}: sem evidência recuperada para caso respondível'
    elif row['expected_case'] == 'out_of_scope':
        assert gate == 'out_of_scope', f'{row["query_id"]}: deveria ser fora de escopo'
    elif row['expected_case'] == 'missing_evidence':
        assert 'sapato' not in corpus_text and 'shoe' not in corpus_text, 'Caso ausente exige inspeção do corpus.'
    return {'query_id': row['query_id'], 'precheck_gate_reason': gate,
            'precheck_source_ids': [item['document_id'] for item in selected]}

prechecks = [precheck_generation_case(row) for row in questions.to_dict(orient='records')]
pd.DataFrame(prechecks).to_csv(EVAL_DIR / 'rag_generation_precheck.csv', index=False, encoding='utf-8')
RUN_GENERATION_EVALUATION = os.environ.get('RUN_GENERATION_EVALUATION', 'false').strip().casefold() in {'1', 'true', 'yes'}
if RUN_GENERATION_EVALUATION:
    results = []
    for row in questions.to_dict(orient='records'):
        record = {'query_id': row['query_id'], 'question': row['question'], 'expected_case': row['expected_case']}
        try:
            baseline = model_without_corpus(row['question'])
            record.update(without_rag_answer=baseline.get('answer'),
                          without_rag_model=baseline.get('model'),
                          without_rag_input_tokens=baseline.get('input_tokens'),
                          without_rag_latency_seconds=baseline.get('latency_seconds'))
        except Exception as error:
            record['without_rag_error'] = type(error).__name__
        try:
            grounded = answer_question(row['question'], generate=True)
            metadata = grounded.get('generation_metadata') or {}
            record.update(rag_answer=grounded.get('answer'), status=grounded['status'],
                          rag_answer_mode=grounded.get('answer_mode'),
                          rag_model=metadata.get('model'),
                          rag_input_tokens=metadata.get('input_tokens'),
                          rag_latency_seconds=metadata.get('latency_seconds'),
                          rag_documents_in_prompt=metadata.get('documents_in_prompt', grounded.get('documents_in_prompt', [])),
                          source_documents=grounded['source_documents'],
                          supporting_excerpts=grounded.get('supporting_excerpts', []))
        except Exception as error:
            record.update(status='generation_error', generation_error=type(error).__name__)
        results.append(record)
    with EVAL_RESULTS_PATH.open('w', encoding='utf-8') as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
    print('Avaliação real opt-in exportada:', EVAL_RESULTS_PATH)
else:
    print('Avaliação real desativada: RUN_GENERATION_EVALUATION=False; nenhuma chamada ao FLAN-T5 foi feita.')

if EVAL_RESULTS_PATH.exists():
    review_rows = []
    for result in read_jsonl(EVAL_RESULTS_PATH):
        review_rows.append({'query_id': result.get('query_id'), 'question': result.get('question'),
                            'status': result.get('status'), 'fidelity_to_text_0_2': '',
                            'relevance_0_2': '', 'completeness_0_2': '',
                            'factual_accuracy_0_2': '', 'appropriate_abstention_0_2': '',
                            'reviewer_notes': '', 'reviewed_by': '', 'review_date': ''})
    review_frame = pd.DataFrame(review_rows)
    human_review_fields = ['fidelity_to_text_0_2', 'relevance_0_2', 'completeness_0_2',
                           'factual_accuracy_0_2', 'appropriate_abstention_0_2',
                           'reviewer_notes', 'reviewed_by', 'review_date']
    if EVAL_REVIEW_PATH.exists():
        existing_review = pd.read_csv(EVAL_REVIEW_PATH, dtype=str, keep_default_na=False)
        existing_by_id = existing_review.set_index('query_id').to_dict(orient='index') if 'query_id' in existing_review else {}
        for index, row in review_frame.iterrows():
            previous = existing_by_id.get(str(row['query_id']), {})
            for field in human_review_fields:
                if previous.get(field, '') != '':
                    review_frame.at[index, field] = previous[field]
    review_frame.to_csv(EVAL_REVIEW_PATH, index=False, encoding='utf-8')
    print('Planilha de revisão humana:', EVAL_REVIEW_PATH)


Avaliação real opt-in exportada: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\evaluation\rag_generation_results.jsonl
Planilha de revisão humana: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\evaluation\rag_manual_review.csv


## 13 · Experimentos ablação e contrafactual (opcionais, isolados da produção)
**Experimento A — Sem contexto**: mede qualitativamente como um LLM responde sem nosso corpus. **Não faz parte do chatbot**.

**Experimento B — Contrafactual sintético**: envia um documento fictício rotulado como teste somente na memória, sem modificar os JSONs reais, para investigar dependência do contexto. Também não altera o recuperador.

Esses experimentos são desativados por padrão e exigem revisão humana para conclusões.

In [18]:
ABLATION_DIR = EVAL_DIR / 'ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
RUN_ABLATION_WITHOUT_CONTEXT = False
RUN_SYNTHETIC_COUNTERFACTUAL = False

if RUN_ABLATION_WITHOUT_CONTEXT:
    sample = 'Qual é a velocidade-base do Pikachu?'
    comparison = {'experiment': 'without_context_vs_rag', 'question': sample,
                  'baseline_without_corpus': model_without_corpus(sample),
                  'rag': answer_question(sample, generate=True),
                  'interpretation': 'Requer revisão humana; não supõe superioridade do RAG.'}
    (ABLATION_DIR / 'without_context_vs_rag.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
if RUN_SYNTHETIC_COUNTERFACTUAL:
    raise RuntimeError('Experimento sintético deve ser implementado somente com um protocolo revisado manualmente.')
print('Experimentos locais opcionais permanecem desativados por padrão.')


Experimentos locais opcionais permanecem desativados por padrão.


## 14 · Relatório de integridade e limitações
O relatório diferencia testes determinísticos locais de geração real. Não declare que a fidelidade do modelo foi comprovada sem executar a API e revisar manualmente a planilha.

In [19]:
report = {
 'project':'Pokédex Inteligente — Trabalho 2', 'notebook':'03_RAG_CHATBOT', 'run_utc':RUN_AT,
 'document_count':len(documents), 'pokemon_count':len(records),
 'retriever':'TF-IDF + cosseno persistido no Notebook 02; sem novo fit',
 'generator':'FLAN-T5 local', 'model':FLAN_MODEL_NAME, 'model_loaded':FLAN_MODEL is not None,
 'input_token_budget':MAX_INPUT_TOKENS, 'minimum_evidence_score':MIN_EVIDENCE_SCORE,
 'structured_fact_validation':True, 'grounded_repair_enabled':True,
 'live_demo_executed':RUN_LIVE_DEMO,
 'generation_evaluation_executed':RUN_GENERATION_EVALUATION,
 'limitations':['FLAN-T5-small pode ser menos consistente em português.',
                'A seleção de fontes e os trechos enviados não comprovam suporte semântico sozinhos.',
                'Tipos, estatísticas, evoluções e formas de obtenção usam reparo fundamentado quando a geração omite valores.',
                'A geração exige que os pesos tenham sido provisionados previamente no cache local.'],
}
report_path=REPORTS_DIR/'rag_chatbot_grounding_report.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
assert actual_sha==manifest['source_document_sha256']
assert index_matrix.shape[0]==len(documents)==len(manifest['document_ids'])
print('Relatório:',report_path)


Relatório: C:\Users\Lucas Sigoli\Documents\Projects\smart-pokedex\data\reports\rag_chatbot_grounding_report.json


## 15 · Síntese para a apresentação
1. **PLN tradicional**: corpus coletado e preparado no Trabalho 1, representado em documentos no Notebook 01 e indexado com TF-IDF no Notebook 02.
2. **Recuperação verificável**: toda pergunta passa pelo recuperador tradicional e pelos controles de escopo/cobertura antes da geração.
3. **LLM como etapa final**: FLAN-T5 formula a resposta a partir do contexto recuperado, sem ferramentas de busca externa; fatos estruturados omitidos acionam um reparo fundamentado e natural, sem documento bruto na resposta.
4. **Fundamentação**: cada resposta aceita traz IDs e os trechos efetivamente enviados; esses dados permitem auditoria, mas não substituem revisão semântica humana.
5. **Avaliação**: Notebook 02 mede recuperação, Notebook 03 mede integridade das fontes, qualidade das respostas e abstenções, com experimentos opcionais de ablação.

**Limitação central:** RAG com verificação de citações não é garantia absoluta de verdade. A validação final depende do conteúdo e da qualidade histórica do corpus.